# 📖 教案：App2 RAG 系统 — 让 LLM 拥有"开卷考试"能力

---

## 课程概览

| 项目 | 内容 |
|:---|:---|
| **课程名称** | RAG（检索增强生成）系统：从原理到完整实现 |
| **对应源码** | `Applications/App2_RAG_System.ipynb` |
| **总时长** | 约 90 分钟（含休息） |
| **目标受众** | 有 Python 基础、了解 LLM 基本概念的学员 |
| **前置知识** | Ch12（RAG 原理）、App1（ReAct Agent 中的 Function Calling 概念） |
| **环境要求** | Ollama 本地后端 或 DashScope API Key；需联网下载 SentenceTransformers 模型（约120MB） |

---

## ⏱ 时间表

| 时间段 | 时长 | 内容 | 对应 Cell |
|:---|:---|:---|:---|
| 00:00–00:08 | 8 min | 开场：什么是 RAG？为什么需要 RAG？ | `intro`, `cdomeyuj1bg` |
| 00:08–00:15 | 7 min | 环境配置与后端选择 | `setup-imports`, `da830dbd`, `setup-backends` |
| 00:15–00:30 | 15 min | 向量数据库实现与文档入库 | `vector-db-header`, `create-vector-store` |
| 00:30–00:45 | 15 min | 向量搜索测试与效果分析 | `search-header`, `test-search` ~ `test-search-4` |
| 00:45–00:48 | 3 min | ☕ 第一次休息 |  |
| 00:48–01:05 | 17 min | 构建完整 RAG 系统（RAGSystem 类） | `rag-header`, `fk3vk5c94du`, `rag-class` |
| 01:05–01:25 | 20 min | RAG 系统测试与边界验证 | `test-rag-header`, `test-rag-1` ~ `test-rag-5` |
| 01:25–01:28 | 3 min | ☕ 第二次休息 |  |
| 01:28–01:40 | 12 min | 高级功能：文档分块与重排序 | `advanced-header`, `chunking` |
| 01:40–01:50 | 10 min | 总结、练习、答疑 | `summary`, `exercise` |

---

## ✅ 课前检查清单

- [ ] Ollama 服务已启动 (`ollama serve`) 或 DashScope API Key 已配置
- [ ] `qwen3:4b` 模型已拉取 (`ollama pull qwen3:4b`)
- [ ] Python 环境中已安装 `sentence-transformers`、`numpy`、`matplotlib`
- [ ] 网络畅通（首次需下载 Embedding 模型约 120MB）
- [ ] 已完成 `PREPARE_OLLAMA.ipynb` 准备步骤
- [ ] 投影/屏幕共享已就绪，字体放大到可读

---

# 第一部分：开场与概念引入（00:00–00:08）

---

## 教学段 1：什么是 RAG？为什么需要它？

📍 **Cell 范围**：`intro`、`cdomeyuj1bg`

⏱ **时间**：8 分钟

🎯 **目标**：让学员理解 RAG 的核心思想——"开卷考试"，建立直觉

---

### 🗣 话术

> 大家好！今天我们来做一个特别实用的项目——RAG 系统。
>
> 先问大家一个问题：**你们考试的时候，是喜欢闭卷还是开卷？**（等2秒）
>
> 对吧，开卷考试总是让人更有信心。为什么？因为你不需要把所有知识都记在脑子里，只需要知道"去哪里翻书、翻哪一页"就行了。
>
> LLM 也面临同样的问题。ChatGPT 的训练数据截止到某个时间点，它不知道你们公司上周发布的产品文档，不知道最新的法律法规。如果你硬问它，它就会"编"——这就是幻觉问题。
>
> **RAG 就是给 LLM 一本"参考书"。** 用户提问时，系统先去知识库里翻找相关内容，把找到的内容连同问题一起交给 LLM，LLM 就可以"看着答案写回答"了。
>
> 大家看 `intro` 这个 Cell 里的流程图：
> ```
> 用户问题 → 向量化查询 → 搜索知识库 → 文档+问题 → LLM生成回答
> ```
> 这就是 RAG 的四步走。今天我们要从零实现每一步。
>
> 在 App1 里我们学了 ReAct Agent 和 Function Calling。RAG 本质上也可以当作一个"工具"——Agent 决定需要查资料时，就调用 RAG 搜索知识库。区别在于，计算器工具返回的是精确答案 `42`，而 RAG 返回的是"相关但可能不完美"的文档片段，LLM 需要从中提炼答案。

---

### 👀 输出要点

- 学员能说出 RAG 的三个核心优势：回答训练数据之外的信息、减少幻觉、知识可实时更新
- 理解"开卷考试"类比
- 明确本节目标：向量数据库 → 向量搜索 → 完整 RAG → 文档分块

---

### ❓ 预判 Q&A

| 问题 | 回答 |
|:---|:---|
| RAG 和 Fine-tuning 有什么区别？ | Fine-tuning 是"把知识背下来"，RAG 是"带着小抄考试"。Fine-tuning 改了模型权重，RAG 不动模型，只是给它更多上下文。二者可以结合。 |
| RAG 能替代 Agent 吗？ | 不能。RAG 解决的是"知识不足"问题，Agent 解决的是"行动能力"问题。实际上 RAG 经常作为 Agent 的一个工具使用。 |
| 为什么不直接把所有文档塞进 Prompt？ | 上下文窗口有限（4K~128K tokens），而知识库可能有几百万文档。而且塞太多无关内容反而会干扰 LLM。 |

---

### ➡️ 转场

> 好，概念清楚了。接下来我们先把环境跑通，确保 Embedding 模型和 LLM 后端都就绪。

---

# 第二部分：环境配置（00:08–00:15）

---

## 教学段 2：环境配置与后端选择

📍 **Cell 范围**：`setup-header`、`setup-imports`、`da830dbd`、`setup-backends`

⏱ **时间**：7 分钟

🎯 **目标**：让学员成功运行环境初始化，理解 Embedding 模型和 LLM 后端的选择逻辑

---

### 🗣 话术

> 现在我们来配环境。RAG 系统需要两个核心组件：
>
> **第一个是 Embedding 模型。** 大家可以把它想象成一个"翻译官"，它把文字翻译成数字向量。为什么要翻译？因为计算机不懂"语义"，但它懂"数字距离"。两段意思相近的文字，翻译成向量后，在向量空间里的距离就很近。
>
> 我们用的是 `paraphrase-multilingual-MiniLM-L12-v2`，这个名字很长，但关键信息是：**多语言**（中英文都支持）、**MiniLM**（小巧高效）。生成的向量是 **384 维**。
>
> 大家运行 `setup-imports` 这个 Cell。（等待）看到 `[OK] 环境准备完成!` 了吗？
>
> 现在运行 `setup-backends`。注意看输出——Embedding 模型会显示 **维度: 384**，这意味着每段文字都会被转成一个 384 个数字组成的数组。
>
> **第二个是 LLM 后端。** 代码会自动按顺序尝试三种后端：Ollama（本地免费）→ DashScope（通义千问云端）→ OpenAI。只要有一个通了就行。大家看看自己的输出，应该显示 `[OK] LLM 使用 Ollama 后端` 或者 `DashScope` 后端。
>
> 如果两个都失败了，按照提示选一种配置就行。推荐 Ollama，因为免费而且数据不出本机。

---

### 👀 输出要点

- `[OK] 环境准备完成!`
- `[OK] Embedding 模型加载成功，维度: 384` — 强调 384 维是这个模型的固定输出
- `[OK] LLM 使用 xxx 后端` — 确认 LLM 可用
- 首次运行会下载模型（约 120MB），后续从 `~/.cache/huggingface/` 缓存加载

---

### ❓ 预判 Q&A

| 问题 | 回答 |
|:---|:---|
| 384 维是什么意思？ | 就是用 384 个浮点数来表示一段文字的"含义"。维度越高，表达能力越强，但计算越慢。384 维是速度和质量的平衡。OpenAI 的 `text-embedding-ada-002` 用 1536 维。 |
| 为什么网络断了还能跑？ | 代码有 fallback 机制——如果 SentenceTransformers 下载失败，会自动回退到 TF-IDF（纯本地，但语义能力弱很多）。 |
| Ollama 和 DashScope 选哪个？ | 教学场景推荐 Ollama：免费、数据不出本地、响应快。生产环境可以用 DashScope 的 `qwen-plus`，效果更好。 |

---

### ➡️ 转场

> 环境通了！我们现在有了"翻译官"（Embedding）和"答题者"（LLM），接下来要做的就是建一个"图书馆"——向量数据库。

---

# 第三部分：向量数据库实现（00:15–00:30）

---

## 教学段 3：创建向量数据库与文档入库

📍 **Cell 范围**：`vector-db-header`、`create-vector-store`

⏱ **时间**：15 分钟

🎯 **目标**：理解向量数据库的核心功能（添加、向量化、搜索），掌握知识库构建流程

---

### 🗣 话术

> 好，现在我们来建"图书馆"。
>
> 传统数据库用关键词搜索——你输入"Transformer"，它就去找包含这个词的文档。这有个问题：如果用户问"注意力机制的模型"，传统搜索可能找不到 Transformer 相关的文档，因为没有完全匹配的关键词。
>
> **向量数据库不一样。** 它把每个文档都变成一个向量（一串数字），搜索时也把查询变成向量，然后比较**向量之间的距离**。意思相近的，距离就近。这就是"语义搜索"。
>
> 打个比方：传统搜索像是在图书馆里"按书名找书"，向量搜索像是"按内容主题找书"——就算书名完全不同，只要内容相关就能找到。
>
> 大家看 `create-vector-store` 这个 Cell。我们准备了 15 个文档，分成四个主题：
> - **architecture**（架构）：Transformer
> - **models**（模型）：GPT、BERT、LLaMA、ChatGPT
> - **training**（训练）：预训练、SFT、RLHF、DPO、LoRA
> - **technical**（技术）：注意力机制、KV Cache、量化、RAG、Agent
>
> 每个文档都有 metadata（元数据），包含主题标签和 ID。这在生产环境中很重要——你可以按主题过滤搜索结果。
>
> 运行这个 Cell，注意看输出：`✓ 已添加 15 个文档，总计 15 个`。这意味着 15 段文字都已经被 Embedding 模型转成了 384 维向量，存进了内存数据库里。

---

### 👀 输出要点

- `✓ 已添加 15 个文档，总计 15 个`
- 强调 `SimpleVectorStore` 是内存版实现，重启就没了
- 注意每个文档有 `metadata`，包含 `topic` 和 `id`，用于后续过滤

---

### ❓ 预判 Q&A

| 问题 | 回答 |
|:---|:---|
| 生产环境用什么向量数据库？ | 常见选择：ChromaDB（轻量、Python 原生）、Milvus（分布式、高性能）、Pinecone（全托管云服务）、Weaviate（带图搜索）。我们这里用内存版是为了教学，原理完全一样。 |
| 15 个文档太少了，真实场景多大？ | 企业 RAG 通常几万到几百万文档。这时候就需要 ANN（近似最近邻）算法，如 HNSW、IVF，用少量精度换巨大速度提升。 |
| metadata 有什么用？ | 生产中可以用 metadata 做过滤：比如用户问法律问题，先按 `topic=legal` 过滤，再做向量搜索，又快又准。 |

---

### ➡️ 转场

> 文档入库了！接下来我们测试搜索效果——看看向量搜索到底准不准。

---

## 教学段 4：向量搜索测试与效果分析

📍 **Cell 范围**：`search-header`、`test-search`、`test-search-2`、`test-search-3`、`test-search-4`

⏱ **时间**：15 分钟

🎯 **目标**：通过四个不同查询，让学员直观感受语义搜索的能力和相似度分数的含义

---

### 🗣 话术

> 现在来"检验图书馆"好不好用。我们做四组搜索实验。
>
> **实验一：** 运行 `test-search`，查询"什么是 Transformer？"。
>
> 看结果——Top 1 相似度 **0.5470**，命中了 Transformer 那篇文档。Top 2 是 BERT（0.5257），因为 BERT 全名里有 Transformer 这个词。Top 3 是 GPT（0.3722），同样包含 Transformer。
>
> 注意这些相似度数值：**余弦相似度范围是 -1 到 1**，0.5 以上就算相当相关了。这个多语言小模型在 0.5 左右能给出准确结果，已经很不错了。
>
> **实验二：** `test-search-2`，查询"如何训练大语言模型？"。
>
> 这次 Top 1 是预训练文档，相似度 **0.6368**——比刚才高。为什么？因为问题和文档在语义上更接近。Top 2 是 SFT（0.4671），这也对，微调确实是训练的一部分。
>
> **大家注意一个细节：** 查询里没有出现"预训练"这三个字，但搜索结果精准命中了。如果用关键词搜索，搜"训练"不一定能搜到"Pre-training"。这就是语义搜索的威力。
>
> **实验三：** `test-search-3`，"LoRA 是什么？如何减少显存？"。
>
> Top 1 相似度 **0.6748**，是目前最高的！LoRA 文档被精准命中。注意这里查询包含两个概念（LoRA + 显存），向量搜索综合理解了整个查询的语义。
>
> **实验四：** `test-search-4`，用英文查询 "What is RLHF and how does it work?"。
>
> 即使知识库是中文文档，英文查询也能找到 RLHF 那篇！相似度 **0.4759**。这就是**多语言 Embedding 模型**的优势——它在同一个向量空间里表示中文和英文，实现跨语言语义匹配。
>
> **互动环节：** 大家觉得四次实验里，哪次搜索最"聪明"？（等3秒让学员讨论）对，实验四最令人惊讶——跨语言搜索，这在企业多语言知识库场景里非常实用。

---

### 👀 输出要点

| 查询 | Top 1 相似度 | 命中文档主题 | 关键观察 |
|:---|:---|:---|:---|
| 什么是 Transformer？ | 0.5470 | architecture | 直接关键词 + 语义双重命中 |
| 如何训练大语言模型？ | 0.6368 | training | 无关键词重叠，纯语义匹配 |
| LoRA + 显存 | 0.6748 | training | 复合查询的综合理解 |
| What is RLHF（英文） | 0.4759 | training | 跨语言语义匹配 |

---

### ❓ 预判 Q&A

| 问题 | 回答 |
|:---|:---|
| 相似度多少算"够好"？ | 没有绝对标准。一般 >0.5 比较可靠，0.3~0.5 看情况，<0.3 基本不相关。生产中通常设一个阈值，低于阈值就不返回。 |
| 为什么英文搜中文相似度偏低？ | 跨语言匹配天然损失一些精度。如果知识库主要是中文，建议用中文查询；多语言支持是"bonus"，不是主力。 |
| Top-K 设多少合适？ | 取决于场景。K=3 是常见默认值。太小可能漏掉相关信息，太大会引入噪音。后面的"重排序"可以缓解这个问题。 |

---

### ➡️ 转场

> 搜索没问题。但光搜到文档还不够——我们需要把文档交给 LLM，让它生成高质量回答。接下来进入核心环节：构建完整的 RAG 系统。

---

# ☕ 第一次休息（00:45–00:48）

---

## 三句话回顾前半段

1. **RAG = 开卷考试**：先检索知识库，再让 LLM "看着资料"回答问题，核心解决"LLM 不知道最新/私有知识"的难题。
2. **向量数据库是"语义图书馆"**：文字被 Embedding 模型翻译成 384 维向量，通过余弦相似度找到语义最相近的文档，而非关键词匹配。
3. **搜索效果验证**：中文查询最高相似度 0.6748（LoRA），英文查中文也能跨语言命中（RLHF，0.4759）。

---

# 第四部分：构建完整 RAG 系统（00:48–01:05）

---

## 教学段 5：RAG 在 Agent 架构中的角色

📍 **Cell 范围**：`rag-header`、`fk3vk5c94du`

⏱ **时间**：5 分钟

🎯 **目标**：将 RAG 与 App1 中学过的 Function Calling 概念联系起来，理解 RAG 作为 Agent 工具的定位

---

### 🗣 话术

> 休息回来！在进入 RAG 类的代码之前，我想先帮大家把知识串起来。
>
> 在 App1 里我们学了 ReAct Agent——LLM 可以调用工具（Function Calling），比如调用计算器、搜索引擎。**RAG 本质上也是一个工具。**
>
> 大家看 `fk3vk5c94du` 这个 Cell 里的 Function Calling 定义——`search_knowledge_base`，参数是 `query` 和 `top_k`。如果把 RAG 集成到 Agent 里，Agent 遇到知识类问题时就会自动调用这个工具。
>
> 但有一个**关键区别**：计算器返回 `42`，是精确答案；RAG 返回的是"相关但可能不完美"的文档片段。所以 LLM 拿到 RAG 结果后，还需要做一次"阅读理解"——从文档里提炼出真正的答案。
>
> 这就是为什么叫"开卷考试"——**开卷不等于抄答案，你还得会看题、会找重点。**

---

### 👀 输出要点

- RAG 可以用 Function Calling 的 JSON Schema 定义为 Agent 工具
- 关键区别：普通工具返回精确结果，RAG 返回"相关文档"，LLM 需二次理解
- 联系 App1 知识点，形成系列课程的知识网络

---

### ❓ 预判 Q&A

| 问题 | 回答 |
|:---|:---|
| RAG 能不能和 ReAct 结合？ | 完全可以！实际上这是最常见的 Agent 架构：Agent 用 ReAct 循环决策，其中一个工具就是 RAG 知识库搜索。App4 多智能体系统会展示更复杂的组合。 |

---

### ➡️ 转场

> 概念串起来了。现在我们来看 RAG 系统的代码实现——RAGSystem 这个类。

---

## 教学段 6：RAGSystem 类逐行精讲

📍 **Cell 范围**：`rag-class`

⏱ **时间**：12 分钟

🎯 **目标**：理解 RAG 三步流程（检索-构建Prompt-生成），掌握 System Prompt 设计和来源追踪

---

### 🗣 话术

> 大家看 `rag-class` 这个 Cell，这是整个 notebook 最核心的代码。我来逐步拆解。
>
> **RAGSystem 类有两个组件：** `vector_store`（向量数据库）和 `llm`（大语言模型）。就像一个学生有"参考书"和"大脑"。
>
> **核心方法是 `answer()`，分三步走：**
>
> **Step 1：检索（retrieve）。** 把用户问题发给向量数据库，拿回 top_k 个最相关的文档。默认 K=3。
>
> **Step 2：构建 Prompt（format_context + SYSTEM_PROMPT）。** 这一步非常关键！大家看 `SYSTEM_PROMPT` 这个模板——它告诉 LLM 四条规则：
> 1. **只用参考资料回答**——防止 LLM 自由发挥编答案
> 2. **没有信息就说没有**——这是减少幻觉的关键指令
> 3. **引用要准确**
> 4. **简洁准确**
>
> 然后把检索到的文档用 `[文档1] [文档2] [文档3]` 的格式拼到 System Prompt 里。LLM 看到的实际上是"一道附带参考资料的阅读理解题"。
>
> **Step 3：LLM 生成回答。** 注意 `temperature=0.3`——比较低，让回答更确定性、更少随机。RAG 场景不需要创意，需要准确。
>
> **还有一个加分方法 `answer_with_sources()`。** 它返回的不只是答案，还有来源列表（哪些文档被用到了、相似度多少）。这在生产环境里很重要——用户可以核实来源，增加信任。
>
> 运行这个 Cell，看到 `[OK] RAG 系统创建成功!` 就可以了。

---

### 👀 输出要点

- `[OK] RAG 系统创建成功!`
- 强调 `SYSTEM_PROMPT` 中"只用参考资料"和"没有就说没有"这两条规则是减少幻觉的核心
- `temperature=0.3` 是 RAG 场景的推荐值
- `answer_with_sources()` 返回 `{"answer": str, "sources": [...]}`，带来源追踪

---

### ❓ 预判 Q&A

| 问题 | 回答 |
|:---|:---|
| temperature 为什么设 0.3 而不是 0？ | 0 意味着完全确定性（贪心解码），有时会导致重复。0.3 允许微小随机性，既准确又自然。如果是创意写作可以设 0.7~1.0。 |
| 如果 LLM 不可用怎么办？ | 代码有 fallback——如果 `self.llm is None`，直接返回检索结果摘要。不生成回答，但搜索功能照常工作。 |
| System Prompt 能自定义吗？ | 当然！不同业务需要不同的 System Prompt。比如法律顾问可以加"请引用具体法条"，客服可以加"请用礼貌口语回答"。 |

---

### ➡️ 转场

> RAG 系统搭好了，现在我们来对它进行全面测试——看看它在不同场景下的表现。

---

# 第五部分：RAG 系统测试（01:05–01:25）

---

## 教学段 7：RAG 系统全面测试

📍 **Cell 范围**：`test-rag-header`、`test-rag-1`、`test-rag-2`、`test-rag-3`、`test-rag-4`、`test-rag-5`

⏱ **时间**：20 分钟

🎯 **目标**：通过 5 个不同场景的测试，验证 RAG 系统的能力边界——正常问答、跨文档推理、带来源回答、知识库外问题

---

### 🗣 话术

> 现在是最有趣的环节——我们来"考一考"我们的 RAG 系统！
>
> **测试 1（`test-rag-1`）：** "什么是 Transformer？它有什么特点？"
>
> 运行后看 verbose 输出。Step 1 检索到 3 个文档，最相关的是 Transformer 架构文档（0.558）。Step 2 把上下文拼好，**327 个字符**——注意这个数字，上下文长度直接影响 LLM 回答质量。Step 3 LLM 生成了一段结构清晰的回答。
>
> 大家注意 LLM 的回答质量——它不是简单复制文档内容，而是**重新组织语言**，加了"核心特点"的总结。这就是"阅读理解"能力。
>
> **测试 2（`test-rag-2`）：** "如何让 Base Model 变成 Chat Model？"
>
> 这个问题需要理解"SFT 是把 Base Model 变成 Chat Model 的关键步骤"这句话。Top 1 命中 SFT 文档（0.426），LLM 准确提取了关键信息。注意 Top 2 是 ChatGPT 文档——因为 ChatGPT 就是一个 Chat Model 的例子，搜索很聪明。
>
> **测试 3（`test-rag-3`）：** "LoRA 和量化如何帮助降低显存？"
>
> 这是一个**跨文档推理**的问题！LoRA 和量化分别在两个不同文档里。看结果——Top 1 是 LoRA（0.733，目前全场最高！），Top 2 是量化（0.534）。LLM 把两个文档的信息综合起来，分别解释了两种方法。**这就是 RAG 的威力——能把分散在多个文档里的信息整合成一个连贯的回答。**
>
> **测试 4（`test-rag-4`）：** 带来源的回答——"RLHF 和 DPO 有什么区别？"
>
> 这次用的是 `answer_with_sources()`。除了回答之外，还返回了 `sources` 列表，每个来源有具体内容、相似度分数和 metadata。在生产系统中，你可以用这些来源做"脚注"，让用户点击查看原文。
>
> **测试 5（`test-rag-5`）：最重要的测试！** "Python 的 GIL 是什么？"
>
> 这个问题在我们的知识库里**完全没有**。看看 RAG 怎么处理——检索到的 3 个文档相似度都很低（最高才 0.358，都是勉强沾边的）。LLM 的回答是：**"参考资料中未提及 Python 的 GIL 相关内容。"**
>
> **这就是我们在 System Prompt 里写的"没有信息就说没有"在起作用！** 如果不加这条规则，LLM 大概率会用自己的训练知识回答 GIL 问题——那就变成"闭卷考试"了，可能会有幻觉。
>
> **互动：** 大家觉得测试 5 更有价值还是测试 3？（等3秒）对，测试 5 验证了系统的**安全边界**。一个好的 RAG 系统不仅要答得对，更要在不确定时敢说"我不知道"。

---

### 👀 输出要点

| 测试 | 问题 | 关键观察 |
|:---|:---|:---|
| test-rag-1 | Transformer 特点 | 上下文 327 字符，LLM 重新组织了语言而非复制 |
| test-rag-2 | Base→Chat Model | SFT 文档准确命中，ChatGPT 文档作为补充 |
| test-rag-3 | LoRA + 量化 | **跨文档推理**：两个文档信息被整合，LoRA 相似度 0.733（全场最高） |
| test-rag-4 | RLHF vs DPO | `answer_with_sources()` 返回来源追踪，生产必备 |
| test-rag-5 | Python GIL | **知识库外问题**：最高相似度仅 0.358，LLM 正确拒绝回答 |

---

### ❓ 预判 Q&A

| 问题 | 回答 |
|:---|:---|
| 相似度低但 LLM 还是回答了怎么办？ | 生产中应加阈值过滤：如果 Top 1 相似度 < 0.3，直接返回"找不到相关信息"，不调用 LLM。省钱且安全。 |
| 跨文档推理有限制吗？ | 有。如果信息分散在 10 个文档里，而 top_k=3 只能拿到 3 个，可能漏掉关键信息。解决办法：增大 K，或用 Multi-hop RAG。 |
| LLM 回答和文档不一致怎么办？ | 这叫"忠实性问题"（faithfulness）。可以用另一个 LLM 做"事实核查"，对比回答和原文是否一致。这是 RAG 评估的重要指标。 |

---

### ➡️ 转场

> 五个测试都通过了！我们的 RAG 系统能准确回答、能跨文档推理、能拒绝不确定的问题。休息一下，回来我们学最后一个高级功能。

---

# ☕ 第二次休息（01:25–01:28）

---

## 三句话回顾

1. **RAG 三步流程**：检索（向量搜索 Top-K）→ 构建 Prompt（System Prompt + 文档上下文）→ LLM 生成回答（temperature=0.3）。
2. **跨文档推理**：LoRA 和量化分别在不同文档，RAG 把它们整合成一个连贯回答；来源追踪让用户可以核实原文。
3. **安全边界**：知识库外问题（Python GIL）被 System Prompt 的"没有就说没有"规则成功拦截，最高相似度仅 0.358。

---

# 第六部分：高级功能（01:28–01:40）

---

## 教学段 8：文档分块——从短文档到长文档的跨越

📍 **Cell 范围**：`advanced-header`、`chunking`

⏱ **时间**：12 分钟

🎯 **目标**：理解为什么需要文档分块、重叠分块的设计原理、句子边界分割的实现

---

### 🗣 话术

> 最后一个高级话题——文档分块。
>
> 之前我们的知识库都是短文档，每篇几十个字。但真实场景中，一份产品文档可能有几万字，一篇论文几千字。直接把整篇文档向量化有两个问题：
>
> 1. **Embedding 精度下降**：文字越长，向量越"模糊"——相当于把一本书的内容压缩成一个 384 维的点，太多信息被丢弃了。
> 2. **上下文浪费**：检索到一篇 5000 字的文档，但用户问的问题可能只和其中一段有关，剩下的 4900 字都是噪音。
>
> **解决方案：分块（Chunking）。** 把长文档切成 200~500 字的小块，每块单独向量化和搜索。
>
> 大家看 `chunking` 这个 Cell 里的 `DocumentChunker` 类。两个关键参数：
> - `chunk_size=200`：每块最多 200 字符
> - `chunk_overlap=50`：相邻块有 50 字符重叠
>
> **为什么要重叠？** 打个比方：如果一句话恰好被切成两半，前半句在块 A，后半句在块 B，两边都理解不了完整意思。重叠确保边界处的信息不会丢失。
>
> 还有一个细节——代码在切割时会**优先在句号、问号等标点处断开**，而不是硬切到 200 字符。这就像你撕纸时沿着折痕撕，比从中间硬撕要整齐得多。
>
> 运行这个 Cell，看输出：原文 **257 字符**被分成了 **2 个块**——块 1 有 181 字符，块 2 有 104 字符。注意块 2 从"RLHF"开始，和块 1 末尾的内容有重叠。
>
> **互动：** 如果 `chunk_size` 设太小会怎样？（等2秒）对，每块太短，信息不完整，搜索召回率会下降。设太大又回到了"整篇文档"的问题。200~500 是常见的经验值。

---

### 👀 输出要点

- 原文 257 字符 → 2 个块（181 字符 + 104 字符）
- 块之间有 50 字符重叠，防止边界信息丢失
- 分割优先在标点符号处断开（`。.！!？?\n`）
- 每个块的 metadata 包含 `chunk_id`、`start`、`end` 位置，可以追溯原文

---

### ❓ 预判 Q&A

| 问题 | 回答 |
|:---|:---|
| chunk_size 怎么选？ | 看 Embedding 模型的最优输入长度。`MiniLM` 建议 128~256 tokens（约 200~500 中文字符）。OpenAI 的 `text-embedding-3-small` 支持更长。经验法则：一个块应该包含一个完整的"知识点"。 |
| 有没有更智能的分块方法？ | 有！语义分块（Semantic Chunking）——根据语义变化自动切割，而不是固定字数。还有基于 Markdown 标题分割、基于段落分割等策略。 |
| 分块后文档数量暴增怎么办？ | 这是正常的。一篇 5000 字的文档按 500 字分块，会变成约 10 个块。向量数据库支持百万级文档，问题不大。搜索时间会略增，但可以用 ANN 索引优化。 |

---

### ➡️ 转场

> 分块是 RAG 从"教学 demo"走向"生产系统"的关键一步。接下来我们做课程总结和练习布置。

---

# 第七部分：总结与练习（01:40–01:50）

---

## 教学段 9：课程总结

📍 **Cell 范围**：`summary`、`exercise`

⏱ **时间**：5 分钟

🎯 **目标**：回顾全课知识点，建立完整的 RAG 知识体系

---

### 🗣 话术

> 我们来做最后的总结。今天从零搭建了一个完整的 RAG 系统，涵盖四大模块：
>
> **1. Embedding 模型**——我们用的 `paraphrase-multilingual-MiniLM-L12-v2`，把文字变成 384 维向量。它是"翻译官"，让计算机能理解语义。
>
> **2. 向量数据库**——`SimpleVectorStore` 存储向量，用余弦相似度做语义搜索。生产环境可以换成 ChromaDB、Milvus 等。
>
> **3. RAG 三步流程**——检索 Top-K 文档 → 构建 System Prompt（附带参考资料）→ LLM 生成回答（temperature=0.3）。关键是 System Prompt 中"只用参考资料"和"没有就说没有"两条规则。
>
> **4. 文档分块**——长文档按 200~500 字符切块，重叠 50 字符防信息丢失，优先在句子边界断开。
>
> **和 App1 的联系**：RAG 可以作为 Agent 的一个 Function Calling 工具，Agent 遇到知识类问题时调用 RAG 搜索知识库。
>
> **和 App3、App4 的展望**：App3（Code Agent）会展示代码生成场景，App4（Multi-Agent）会展示多个 Agent 协作，其中可能包含一个专门的"知识检索 Agent"。

---

### 👀 输出要点

- 四大模块：Embedding → 向量数据库 → RAG 流程 → 文档分块
- 核心数值回顾：384 维、15 个文档、余弦相似度 0.3~0.7 区间、temperature=0.3
- 与系列课程的关联：App1（Function Calling）→ App2（RAG）→ App3/App4

---

## 教学段 10：练习布置

⏱ **时间**：5 分钟

---

### 练习 1：添加相似度阈值过滤（难度：低）

**目标**：修改 `RAGSystem.answer()` 方法，当 Top 1 相似度低于 0.3 时，直接返回"找不到相关信息"，不调用 LLM。

**提示节奏**：
1. 先想想在哪一步加过滤（Step 1 之后、Step 2 之前）
2. 提示：`results[0]['score']` 就是 Top 1 的相似度
3. 最终代码：在 `if not results:` 之后加一个 `if results[0]['score'] < threshold:` 判断

**常见错误**：
- 忘记 `results` 可能为空，在空列表上取 `[0]` 会报 `IndexError`
- 阈值设太高（如 0.8），导致几乎所有查询都被拒绝

**验证方法**：用 "Python 的 GIL 是什么？" 测试，应该直接返回拒绝消息而不调用 LLM

---

### 练习 2：用自己的文档构建知识库（难度：中）

**目标**：使用 `exercise` Cell 中的 `load_markdown_file()` 加载一个真实 Markdown 文件，然后用 `DocumentChunker` 分块入库，最后测试 RAG 问答。

**提示节奏**：
1. 先准备一个 `.md` 文件（可以用课程的 README 或任何文档）
2. 用 `load_markdown_file()` 加载并按 `##` 标题分割
3. 用 `DocumentChunker` 进一步分块（如果段落很长）
4. 用 `vector_store.add_documents()` 入库
5. 用 `rag.answer()` 测试

**常见错误**：
- 文件编码问题：确保文件是 UTF-8
- 忘记分块直接入库，导致长段落搜索效果差

**验证方法**：对自己文档中的内容提问，看回答是否准确引用了原文

---

### 练习 3：实现混合检索（难度：高）

**目标**：结合关键词搜索（BM25/TF-IDF）和向量语义搜索，取两者结果的并集，用加权分数排序。

**提示节奏**：
1. 理解两种搜索的互补性：关键词搜精确匹配，向量搜语义相似
2. 分别获取两种搜索的 Top-K 结果
3. 用加权公式合并：`final_score = alpha * semantic_score + (1-alpha) * keyword_score`，`alpha=0.7` 是常见起点
4. 去重并按 `final_score` 排序

**常见错误**：
- 两种分数的量纲不同，需要先归一化（0~1 范围）
- 忘记去重——同一个文档可能同时被两种搜索命中

---

# 附录

---

## 附录 A：时间表速查

| 段落 | 时间 | 内容 | 核心 Cell |
|:---|:---|:---|:---|
| 1 | 00:00–00:08 (8min) | RAG 概念："开卷考试"类比 | `intro`, `cdomeyuj1bg` |
| 2 | 00:08–00:15 (7min) | 环境配置：Embedding + LLM 后端 | `setup-imports`, `setup-backends` |
| 3 | 00:15–00:30 (15min) | 向量数据库：15 文档入库 | `create-vector-store` |
| 4 | 00:30–00:45 (15min) | 搜索测试：4 组实验 | `test-search` ~ `test-search-4` |
| 休息 | 00:45–00:48 (3min) | 三句话回顾 | — |
| 5 | 00:48–00:53 (5min) | RAG 与 Agent 关系 | `fk3vk5c94du` |
| 6 | 00:53–01:05 (12min) | RAGSystem 类精讲 | `rag-class` |
| 7 | 01:05–01:25 (20min) | RAG 五组测试 | `test-rag-1` ~ `test-rag-5` |
| 休息 | 01:25–01:28 (3min) | 三句话回顾 | — |
| 8 | 01:28–01:40 (12min) | 文档分块 | `chunking` |
| 9-10 | 01:40–01:50 (10min) | 总结 + 练习布置 | `summary`, `exercise` |

---

## 附录 B：关键数据速查

| 数据 | 值 | 出处 |
|:---|:---|:---|
| Embedding 模型 | `paraphrase-multilingual-MiniLM-L12-v2` | `setup-backends` |
| 向量维度 | 384 | `setup-backends` 输出 |
| 知识库文档数 | 15 | `create-vector-store` 输出 |
| 文档主题分类 | architecture, models, training, technical | `create-vector-store` metadata |
| 最高搜索相似度 | 0.7330（LoRA + 量化查询） | `test-rag-3` |
| 最低命中相似度 | 0.3580（GIL 查询，知识库外） | `test-rag-5` |
| RAG temperature | 0.3 | `rag-class` answer() 方法 |
| 分块大小 | 200 字符，重叠 50 | `chunking` |
| 分块测试结果 | 257 字符 → 2 块（181 + 104） | `chunking` 输出 |
| 模型下载大小 | 约 120MB | `da830dbd` 说明 |

---

## 附录 C：应急预案

| 故障场景 | 现象 | 解决方案 |
|:---|:---|:---|
| Ollama 未启动 | `[!] Ollama 连接失败` | 新终端运行 `ollama serve`，等待启动后重新运行 `setup-backends` |
| 模型未下载 | Ollama 连接成功但 chat 失败 | 运行 `ollama pull qwen3:4b`（约 2.5GB），等待完成 |
| 网络不可用 | SentenceTransformers 下载失败 | 代码自动回退到 TF-IDF；提前告知学员效果会略差 |
| DashScope Key 无效 | `[!] DashScope 不可用` | 检查 Key 是否正确，或切换到 Ollama |
| 内存不足 | Embedding 加载时 OOM | 关闭其他应用释放内存；或使用更小的模型 |
| LLM 全部不可用 | `[X] 请配置 LLM 后端` | RAG 仍可运行搜索功能，只是不调用 LLM 生成；展示 fallback 逻辑 |
| 搜索结果不理想 | 相似度偏低 | 检查文档是否成功入库（数量是否为 15）；确认 Embedding 模型加载正常 |
| 课堂节奏过快/过慢 | 学员跟不上或太闲 | 快：跳过 `test-search-3`/`test-search-4`；慢：让学员自定义查询测试 |

---

## 附录 D：板书/白板要点

课前可在白板上预先画好以下内容：

```
┌─────────────────────────────────────────────────────────┐
│                    RAG 系统架构                           │
│                                                          │
│  用户问题 ──→ Embedding ──→ 向量搜索 ──→ Top-K 文档     │
│                               │                          │
│                         向量数据库                        │
│                     (15 篇 × 384 维)                     │
│                                                          │
│  Top-K 文档 ──→ System Prompt ──→ LLM ──→ 回答          │
│                 "只用参考资料"      t=0.3                 │
│                 "没有就说没有"                             │
└─────────────────────────────────────────────────────────┘
```